# Cleaning_CYRV_sellers_dataset


## Conclusion

The dataset was inspected for missing values, duplicate records, duplicate seller IDs, ZIP code formatting, invalid state codes, city formatting inconsistencies, and geographic mismatches.

Seller city values were cross-validated against `CYRV_geolocation_dataset.csv` using ZIP code prefix and state. A total of 30 seller records with confirmed formatting or location issues were standardized. No records were removed, and ambiguous geographic mismatches were retained because the available reference data was insufficient to determine which source value was incorrect.

The cleaned dataset retains all 3,095 original seller records, with unique seller IDs, no missing values, and no duplicate rows, and is ready for downstream analysis and integration with related datasets.

## Data Cleaning Summary

| Check | Result | Decision |
|---|---|---|
| Dataset shape | 3,095 rows × 4 columns | Preserved |
| Missing values | 0 | No action required |
| Exact duplicate rows | 0 | No action required |
| Seller ID uniqueness | 3,095 unique IDs; 0 duplicated IDs | Preserved |
| ZIP code prefixes | 1,027 four-digit and 2,068 five-digit values | Original `int64` representation retained for compatibility with related datasets |
| Seller states | 23 unique state codes | Preserved |
| State formatting | 0 leading/trailing spaces, 0 non-uppercase values, 0 invalid code lengths, 0 empty values | No action required |
| Seller cities | 611 unique values before cleaning | Investigated for formatting and geographic inconsistencies |
| City leading/trailing spaces | 0 | No action required |
| City multiple spaces | 3 rows before cleaning; 0 after cleaning | Standardized to single spaces |
| City formatting and invalid values | Confirmed inconsistencies identified | Standardized only where the intended city could be reliably determined |
| Confirmed city corrections | 30 seller records | Corrected using explicit mappings and ZIP/state cross-validation with the geolocation dataset |
| Geographic cross-validation | 2,966 matches, 87 mismatches, 42 ZIP/state combinations not found | Mismatches were not automatically corrected without sufficient evidence |
| Ambiguous geographic values | Conflicting or non-unique reference information remained for some records | Retained unchanged |
| Special-character city values after cleaning | 4 unique values remain | Retained because they are valid or could not be reliably corrected |
| Row removal | 0 rows | All 3,095 seller records retained |
| Saved-file validation | Shape, missing values, duplicates, IDs, and data types match expected results | Validation passed |

**Output:** `CYRV_sellers_cleaned.csv`

In [52]:
import pandas as pd

sellers = pd.read_csv("/Users/yuliiapotrymai/Desktop/SpikupCapstone2026_CYRV/data/cyrv/CYRV_sellers_dataset.csv")

sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [53]:
print("Shape:", sellers.shape)

print("\nData types:")
print(sellers.dtypes)

Shape: (3095, 4)

Data types:
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object


In [3]:
missing_values = sellers.isna().sum()

missing_percent = (
    sellers.isna().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percent": missing_percent
}).sort_values("missing_count", ascending=False)

missing_summary


,missing_count,missing_percent
seller_id,0,0.0
seller_zip_code_prefix,0,0.0
seller_city,0,0.0
seller_state,0,0.0


In [4]:
print("Duplicate rows:", sellers.duplicated().sum())

Duplicate rows: 0


In [5]:
print("Total rows:", len(sellers))
print("Unique seller IDs:", sellers["seller_id"].nunique())
print("Duplicated seller IDs:", sellers["seller_id"].duplicated().sum())


Total rows: 3095
Unique seller IDs: 3095
Duplicated seller IDs: 0


In [6]:
print("Min ZIP prefix:", sellers["seller_zip_code_prefix"].min())
print("Max ZIP prefix:", sellers["seller_zip_code_prefix"].max())
print("Unique ZIP prefixes:", sellers["seller_zip_code_prefix"].nunique())


Min ZIP prefix: 1001
Max ZIP prefix: 99730
Unique ZIP prefixes: 2246


In [7]:
zip_length_counts = (
    sellers["seller_zip_code_prefix"]
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

zip_length_counts

seller_zip_code_prefix
4    1027
5    2068
Name: count, dtype: int64

In [8]:
four_digit_zips = sellers[
    sellers["seller_zip_code_prefix"].astype(str).str.len() == 4
]

print("4-digit ZIP rows:", len(four_digit_zips))
print("Min:", four_digit_zips["seller_zip_code_prefix"].min())
print("Max:", four_digit_zips["seller_zip_code_prefix"].max())

four_digit_zips.head(10)

4-digit ZIP rows: 1027
Min: 1001
Max: 9981


,seller_id,seller_zip_code_prefix,seller_city,seller_state
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
8,768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP
12,8bd0f31cf0a614c658f6763bd02dea69,1222,sao paulo,SP
13,05a48cc8859962767935ab9087417fbb,5372,sao paulo,SP
19,f9ec7093df3a7b346b7bcf7864069ca3,5138,sao paulo,SP
22,f7496d659ca9fdaf323c0aae84176632,4156,sao paulo,SP
27,116ccb1a1604bc88e4d234a8c23f33de,9850,sao bernardo do campo,SP
28,430315b7bb4b6e4b3c978f9dfa9b0558,4857,sao paulo,SP
31,e9e446d01bd10a97a8ffcfc4a3a20cb2,2261,sao paulo,SP
33,d9a84e1403de8da0c3aa531d6d108ba6,3562,sao paulo,SP


In [9]:
print("Unique states:", sellers["seller_state"].nunique())

sellers["seller_state"].value_counts().sort_index()

Unique states: 23


seller_state
AC       1
AM       1
BA      19
CE      13
DF      30
ES      23
GO      40
MA       1
MG     244
MS       5
MT       4
PA       1
PB       6
PE       9
PI       1
PR     349
RJ     171
RN       5
RO       2
RS     129
SC     190
SE       2
SP    1849
Name: count, dtype: int64

In [10]:
state = sellers["seller_state"]

print(
    "Leading/trailing spaces:",
    state.ne(state.str.strip()).sum()
)

print(
    "Non-uppercase values:",
    state.ne(state.str.upper()).sum()
)

print(
    "Invalid state code length:",
    state.str.len().ne(2).sum()
)

print(
    "Empty/whitespace-only:",
    state.str.strip().eq("").sum()
)

Leading/trailing spaces: 0
Non-uppercase values: 0
Invalid state code length: 0
Empty/whitespace-only: 0


In [11]:
city = sellers["seller_city"]

print("Unique cities:", city.nunique())

print(
    "Leading/trailing spaces:",
    city.ne(city.str.strip()).sum()
)

print(
    "Contains uppercase:",
    city.str.contains(r"[A-Z]", regex=True).sum()
)

print(
    "Empty/whitespace-only:",
    city.str.strip().eq("").sum()
)

Unique cities: 611
Leading/trailing spaces: 0
Contains uppercase: 0
Empty/whitespace-only: 0


In [12]:
city.value_counts().head(20)

seller_city
sao paulo                694
curitiba                 127
rio de janeiro            96
belo horizonte            68
ribeirao preto            52
guarulhos                 50
ibitinga                  49
santo andre               45
campinas                  41
maringa                   40
sao jose do rio preto     33
sao bernardo do campo     32
osasco                    32
sorocaba                  32
brasilia                  28
porto alegre              28
londrina                  26
goiania                   23
joinville                 22
blumenau                  21
Name: count, dtype: int64

In [13]:
print(
    "Multiple spaces:",
    city.str.contains(r"\s{2,}", regex=True).sum()
)

print(
    "Contains special characters:",
    city.str.contains(r"[^a-z\s]", regex=True).sum()
)

Multiple spaces: 3
Contains special characters: 34


In [14]:
special_city_names = sorted(
    city[
        city.str.contains(r"[^a-z\s]", regex=True)
    ].unique()
)

special_city_names

['04482255',
 'andira-pr',
 "arraial d'ajuda (porto seguro)",
 'auriflama/sp',
 'barbacena/ minas gerais',
 'carapicuiba / sao paulo',
 'cariacica / es',
 'jacarei / sao paulo',
 'lages - sc',
 'maua/sao paulo',
 'mogi das cruzes / sp',
 'novo hamburgo, rio grande do sul, brasil',
 'pinhais/pr',
 'ribeirao preto / sao paulo',
 'rio de janeiro / rio de janeiro',
 'rio de janeiro \\rio de janeiro',
 'rio de janeiro, rio de janeiro, brasil',
 "santa barbara d'oeste",
 'santa barbara d´oeste',
 'santo andre/sao paulo',
 "sao miguel d'oeste",
 'sao paulo - sp',
 'sao paulo / sao paulo',
 'sao sebastiao da grama/sp',
 'são paulo',
 'sbc/sp',
 'sp / sp',
 'vendas@creditparts.com.br']

In [15]:
city_issues = sellers[
    sellers["seller_city"].str.contains(
        r"[^a-z\s]",
        regex=True
    )
].sort_values(["seller_state", "seller_city"])

print("Rows with special characters:", len(city_issues))

city_issues

Rows with special characters: 34


,seller_id,seller_zip_code_prefix,seller_city,seller_state
874,4aba391bc3b88717ce08eb11e44937b2,45816,arraial d'ajuda (porto seguro),BA
622,7994b065a7ffb14e71c6312cf87b9de2,29142,cariacica / es,ES
1447,fe9d9cf8631285d5982c6e2cf27fb114,36200,barbacena/ minas gerais,MG
1610,20cb7c2fde3e5bf10f0bbe7394e1c6a9,86385,andira-pr,PR
1712,6025c79c035c3d772133b8b8238463b2,83327,pinhais/pr,PR
2258,4b5f66b7adcf57f1ecc0d3c07dd6b177,87025,vendas@creditparts.com.br,PR
517,ceb7b4fb9401cd378de7886317ad1b47,22790,04482255,RJ
1649,d79e8478eed9999493990b44955fb22e,20081,rio de janeiro / rio de janeiro,RJ
1346,cf1313c6e2c01c2f4b014f97db4bcd2b,22050,rio de janeiro \rio de janeiro,RJ
2988,f9eedec3129e8cc6b6429c42d0808c5b,22793,"rio de janeiro, rio de janeiro, brasil",RJ


In [16]:
sellers[
    sellers["seller_city"].str.contains(
        r"\s{2,}",
        regex=True
    )
]

,seller_id,seller_zip_code_prefix,seller_city,seller_state
191,e88165a185134e13fdfc85d4fa654db8,8517,ferraz de vasconcelos,SP
576,8a1ff5c35f6595a73fef4c7b96e4908a,83091,sao jose dos pinhais,PR
1705,ea566164622c6b439516ab18062c42cd,5303,sao paulo,SP


In [17]:
issue_zips = city_issues["seller_zip_code_prefix"].unique()

zip_city_reference = (
    sellers[
        sellers["seller_zip_code_prefix"].isin(issue_zips)
    ]
    .groupby(["seller_zip_code_prefix", "seller_state"])["seller_city"]
    .agg(lambda x: sorted(set(x)))
    .reset_index()
)

zip_city_reference

,seller_zip_code_prefix,seller_state,seller_city
0,3363,SP,[sp / sp]
1,3407,SP,[sao paulo / sao paulo]
2,4007,SP,[sao paulo - sp]
3,4130,SP,[sao paulo - sp]
4,4557,SP,[são paulo]
5,5353,SP,[sao paulo - sp]
6,6311,SP,[carapicuiba / sao paulo]
7,8717,SP,[mogi das cruzes / sp]
8,9230,SP,"[santo andre, santo andre/sao paulo]"
9,9380,SP,"[maua, maua/sao paulo]"


In [18]:
geolocation = pd.read_csv(
    "../../data/cyrv/CYRV_geolocation_dataset.csv"
)

print("Shape:", geolocation.shape)

geolocation.head()

Shape: (1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [19]:
geolocation[
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ]
].head()

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
0,1037,sao paulo,SP
1,1046,sao paulo,SP
2,1046,sao paulo,SP
3,1041,sao paulo,SP
4,1035,sao paulo,SP


In [20]:
issue_zips = city_issues["seller_zip_code_prefix"].unique()

In [21]:
geo_city_reference = (
    geolocation[
        geolocation["geolocation_zip_code_prefix"].isin(issue_zips)
    ]
    .groupby(
        ["geolocation_zip_code_prefix", "geolocation_state"]
    )["geolocation_city"]
    .agg(lambda x: sorted(set(x)))
    .reset_index()
)

geo_city_reference

,geolocation_zip_code_prefix,geolocation_state,geolocation_city
0,3363,SP,"[sao paulo, são paulo]"
1,3407,SP,"[sao paulo, são paulo]"
2,4007,SP,"[sao paulo, são paulo]"
3,4130,SP,"[sao paulo, são paulo]"
4,4557,SP,"[sao paulo, são paulo]"
5,5353,SP,"[sao paulo, são paulo]"
6,6311,SP,"[carapicuiba, carapicuíba]"
7,8717,SP,[mogi das cruzes]
8,9230,SP,"[santo andre, santo andré]"
9,9380,SP,"[maua, mauá]"


In [24]:
import unicodedata

def normalize_city(city):
    city = str(city).lower().strip()

    # Remove accents / diacritics
    city = "".join(
        char for char in unicodedata.normalize("NFKD", city)
        if not unicodedata.combining(char)
    )

    # Normalize multiple spaces
    city = " ".join(city.split())

    return city

In [25]:
sellers_validation = sellers.copy()
geo_validation = geolocation.copy()

sellers_validation["city_normalized"] = (
    sellers_validation["seller_city"].apply(normalize_city)
)

geo_validation["city_normalized"] = (
    geo_validation["geolocation_city"].apply(normalize_city)
)

In [26]:
geo_reference = (
    geo_validation
    .groupby(
        ["geolocation_zip_code_prefix", "geolocation_state"]
    )["city_normalized"]
    .agg(set)
    .reset_index(name="reference_cities")
)

geo_reference.head()


,geolocation_zip_code_prefix,geolocation_state,reference_cities
0,1001,SP,{sao paulo}
1,1002,SP,{sao paulo}
2,1003,SP,{sao paulo}
3,1004,SP,{sao paulo}
4,1005,SP,{sao paulo}


In [27]:
seller_geo_check = sellers_validation.merge(
    geo_reference,
    how="left",
    left_on=["seller_zip_code_prefix", "seller_state"],
    right_on=["geolocation_zip_code_prefix", "geolocation_state"]
)

In [28]:
seller_geo_check["geo_match"] = seller_geo_check.apply(
    lambda row:
        pd.NA
        if not isinstance(row["reference_cities"], set)
        else row["city_normalized"] in row["reference_cities"],
    axis=1
)

In [29]:
print("Total sellers:", len(seller_geo_check))

print(
    "Matched:",
    (seller_geo_check["geo_match"] == True).sum()
)

print(
    "Mismatched:",
    (seller_geo_check["geo_match"] == False).sum()
)

print(
    "ZIP/state not found in geolocation:",
    seller_geo_check["geo_match"].isna().sum()
)

Total sellers: 3095
Matched: 2966
Mismatched: 87
ZIP/state not found in geolocation: 42


In [30]:
city_mismatches = seller_geo_check[
    seller_geo_check["geo_match"] == False
][
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "reference_cities"
    ]
]

city_mismatches

,seller_id,seller_zip_code_prefix,seller_city,seller_state,reference_cities
29,406822777a0b9eb5c50e442dd4cd3ec5,18500,tatui,SP,{laranjal paulista}
43,5c030029b5916fed0986310385ec9009,88075,sao jose,SC,{florianopolis}
78,731ef20c231d9a7103a425e83fd91271,88501,lages - sc,SC,{lages}
79,78813699ffac347fe27dba345a5f1551,95711,bento goncalves,RS,{vale dos vinhedos}
103,99cd94252748d2bdde08e17858233602,12401,sao paulo,SP,{pindamonhangaba}
...,...,...,...,...,...
2946,06579cb253ecd5a3a12a9e6eb6bf8f47,4007,sao paulo - sp,SP,{sao paulo}
2988,f9eedec3129e8cc6b6429c42d0808c5b,22793,"rio de janeiro, rio de janeiro, brasil",RJ,{rio de janeiro}
3015,7f5e4d5efad7e44b91115dd1decb65f3,12306,jacarei / sao paulo,SP,{jacarei}
3026,17e34d8224d27a541263c4c64b11a56b,14085,riberao preto,SP,{ribeirao preto}


In [31]:
import re

def clean_city_format(city):
    city = normalize_city(city)

    # Remove country/state information after separators
    city = re.split(r"\s*/\s*|\s*-\s*|\\|,", city)[0]

    # Normalize spaces again
    city = " ".join(city.split())

    return city

city_issues_test = city_issues.copy()

city_issues_test["proposed_city"] = (
    city_issues_test["seller_city"]
    .apply(clean_city_format)
)

city_issues_test[
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "proposed_city"
    ]
]

,seller_zip_code_prefix,seller_city,seller_state,proposed_city
874,45816,arraial d'ajuda (porto seguro),BA,arraial d'ajuda (porto seguro)
622,29142,cariacica / es,ES,cariacica
1447,36200,barbacena/ minas gerais,MG,barbacena
1610,86385,andira-pr,PR,andira
1712,83327,pinhais/pr,PR,pinhais
2258,87025,vendas@creditparts.com.br,PR,vendas@creditparts.com.br
517,22790,04482255,RJ,04482255
1649,20081,rio de janeiro / rio de janeiro,RJ,rio de janeiro
1346,22050,rio de janeiro \rio de janeiro,RJ,rio de janeiro
2988,22793,"rio de janeiro, rio de janeiro, brasil",RJ,rio de janeiro


In [32]:
city_mapping = {
    "cariacica / es": "cariacica",
    "barbacena/ minas gerais": "barbacena",
    "pinhais/pr": "pinhais",

    "rio de janeiro / rio de janeiro": "rio de janeiro",
    "rio de janeiro \\rio de janeiro": "rio de janeiro",
    "rio de janeiro, rio de janeiro, brasil": "rio de janeiro",

    "novo hamburgo, rio grande do sul, brasil": "novo hamburgo",
    "lages - sc": "lages",

    "auriflama/sp": "auriflama",
    "carapicuiba / sao paulo": "carapicuiba",
    "jacarei / sao paulo": "jacarei",
    "maua/sao paulo": "maua",
    "mogi das cruzes / sp": "mogi das cruzes",
    "ribeirao preto / sao paulo": "ribeirao preto",
    "santo andre/sao paulo": "santo andre",

    "sao paulo - sp": "sao paulo",
    "sao paulo / sao paulo": "sao paulo",
    "são paulo": "sao paulo",

    "sao sebastiao da grama/sp": "sao sebastiao da grama"
}

In [33]:
rows_to_clean = sellers[
    sellers["seller_city"].isin(city_mapping)
].copy()

print("Rows to be cleaned:", len(rows_to_clean))
print("Unique original city values:", rows_to_clean["seller_city"].nunique())

Rows to be cleaned: 20
Unique original city values: 18


In [34]:
cleaning_preview = rows_to_clean[
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
].copy()

cleaning_preview["cleaned_city"] = (
    cleaning_preview["seller_city"].replace(city_mapping)
)

cleaning_preview.sort_values(
    ["seller_state", "seller_city"]
)

,seller_zip_code_prefix,seller_city,seller_state,cleaned_city
622,29142,cariacica / es,ES,cariacica
1447,36200,barbacena/ minas gerais,MG,barbacena
1712,83327,pinhais/pr,PR,pinhais
1649,20081,rio de janeiro / rio de janeiro,RJ,rio de janeiro
1346,22050,rio de janeiro \rio de janeiro,RJ,rio de janeiro
2988,22793,"rio de janeiro, rio de janeiro, brasil",RJ,rio de janeiro
551,93310,"novo hamburgo, rio grande do sul, brasil",RS,novo hamburgo
78,88501,lages - sc,SC,lages
237,15350,auriflama/sp,SP,auriflama
2162,6311,carapicuiba / sao paulo,SP,carapicuiba


In [35]:
mapping_keys_not_found = [
    city_name
    for city_name in city_mapping
    if city_name not in sellers["seller_city"].values
]

print("Mapping keys not found:", len(mapping_keys_not_found))

mapping_keys_not_found


Mapping keys not found: 1


['são paulo']

In [36]:
sellers[
    sellers["seller_city"].isin([
        "04482255",
        "vendas@creditparts.com.br",
        "sbc/sp",
        "sp / sp"
    ])
][
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
]

,seller_zip_code_prefix,seller_city,seller_state
517,22790,04482255,RJ
869,9726,sbc/sp,SP
1004,3363,sp / sp,SP
2258,87025,vendas@creditparts.com.br,PR


In [37]:
city_mapping = {
    # City + state/country formatting
    "cariacica / es": "cariacica",
    "barbacena/ minas gerais": "barbacena",
    "pinhais/pr": "pinhais",

    "rio de janeiro / rio de janeiro": "rio de janeiro",
    "rio de janeiro \\rio de janeiro": "rio de janeiro",
    "rio de janeiro, rio de janeiro, brasil": "rio de janeiro",

    "novo hamburgo, rio grande do sul, brasil": "novo hamburgo",
    "lages - sc": "lages",

    "auriflama/sp": "auriflama",
    "carapicuiba / sao paulo": "carapicuiba",
    "jacarei / sao paulo": "jacarei",
    "maua/sao paulo": "maua",
    "mogi das cruzes / sp": "mogi das cruzes",
    "ribeirao preto / sao paulo": "ribeirao preto",
    "santo andre/sao paulo": "santo andre",

    "sao paulo - sp": "sao paulo",
    "sao paulo / sao paulo": "sao paulo",
    "são paulo": "sao paulo",

    "sao sebastiao da grama/sp": "sao sebastiao da grama",

    # Invalid/non-standard city values confirmed using ZIP + state
    "04482255": "rio de janeiro",
    "vendas@creditparts.com.br": "maringa",
    "sbc/sp": "sao bernardo do campo",
    "sp / sp": "sao paulo",

    # Inconsistent apostrophe
    "santa barbara d´oeste": "santa barbara d'oeste",

    # Multiple spaces
    "ferraz de  vasconcelos": "ferraz de vasconcelos",
    "sao  jose dos pinhais": "sao jose dos pinhais",
    "sao  paulo": "sao paulo"
}

In [38]:
rows_to_clean = sellers[
    sellers["seller_city"].isin(city_mapping)
].copy()

print("Rows to be cleaned:", len(rows_to_clean))
print("Unique original city values:", rows_to_clean["seller_city"].nunique())

Rows to be cleaned: 29
Unique original city values: 26


In [39]:
rows_to_clean[
    rows_to_clean["seller_city"].isin([
        "04482255",
        "vendas@creditparts.com.br",
        "sbc/sp",
        "sp / sp"
    ])
][
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
]

,seller_zip_code_prefix,seller_city,seller_state
517,22790,04482255,RJ
869,9726,sbc/sp,SP
1004,3363,sp / sp,SP
2258,87025,vendas@creditparts.com.br,PR


In [40]:
cleaning_preview = rows_to_clean[
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
].copy()

cleaning_preview["cleaned_city"] = (
    cleaning_preview["seller_city"].replace(city_mapping)
)

cleaning_preview.sort_values(
    ["seller_state", "seller_city"]
)

,seller_zip_code_prefix,seller_city,seller_state,cleaned_city
622,29142,cariacica / es,ES,cariacica
1447,36200,barbacena/ minas gerais,MG,barbacena
1712,83327,pinhais/pr,PR,pinhais
576,83091,sao jose dos pinhais,PR,sao jose dos pinhais
2258,87025,vendas@creditparts.com.br,PR,maringa
517,22790,04482255,RJ,rio de janeiro
1649,20081,rio de janeiro / rio de janeiro,RJ,rio de janeiro
1346,22050,rio de janeiro \rio de janeiro,RJ,rio de janeiro
2988,22793,"rio de janeiro, rio de janeiro, brasil",RJ,rio de janeiro
551,93310,"novo hamburgo, rio grande do sul, brasil",RS,novo hamburgo


In [41]:
sellers.loc[
    sellers["seller_zip_code_prefix"] == 4557,
    ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]
]

,seller_id,seller_zip_code_prefix,seller_city,seller_state
360,a3fa18b3f688ec0fca3eb8bfcbd2d5b3,4557,são paulo,SP


In [42]:
repr(
    sellers.loc[
        sellers["seller_zip_code_prefix"] == 4557,
        "seller_city"
    ].iloc[0]
)

"'são paulo'"

In [43]:
unicode_city_case = sellers[
    (sellers["seller_zip_code_prefix"] == 4557) &
    (sellers["seller_state"] == "SP") &
    (sellers["seller_city"].apply(normalize_city) == "sao paulo")
][
    ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]
].copy()

unicode_city_case["cleaned_city"] = "sao paulo"

unicode_city_case

,seller_id,seller_zip_code_prefix,seller_city,seller_state,cleaned_city
360,a3fa18b3f688ec0fca3eb8bfcbd2d5b3,4557,são paulo,SP,sao paulo


## Cleaning Decisions

Based on the inspection, validation, and cross-validation with the geolocation dataset, the following cleaning decisions were made:

- **Missing values:** No missing values were found. No imputation is required.
- **Duplicate rows:** No exact duplicate rows were found. No rows will be removed.
- **Seller IDs:** All 3,095 `seller_id` values are unique. No duplicate seller identifiers were found.
- **ZIP code prefixes:** Both 4-digit and 5-digit representations are present because the column is stored as `int64`, which does not preserve leading zeros. The original representation will be retained to maintain compatibility with related datasets.
- **Seller states:** All state codes are uppercase, two characters long, and contain no leading/trailing spaces or empty values. No cleaning is required.
- **Seller cities:** Formatting inconsistencies and invalid city values were identified. Confirmed cases were cross-validated against `CYRV_geolocation_dataset.csv` using ZIP code prefix and state.
- **Confirmed city corrections:** 30 seller records will be standardized where the intended city could be reliably determined. This includes city/state combinations, duplicated geographic information, invalid values such as an email address or numeric string, inconsistent spacing, and one Unicode variant.
- **Geographic mismatches:** 87 seller records did not exactly match the geolocation reference after basic normalization. These records will not be automatically corrected because a mismatch alone does not establish whether the city or ZIP code is incorrect.
- **Missing geolocation references:** 42 seller ZIP/state combinations were not found in the geolocation dataset. These records will be retained unchanged.
- **Ambiguous cases:** Values with conflicting or ambiguous geographic evidence, such as `andira-pr` and `arraial d'ajuda (porto seguro)`, will be retained rather than modified without sufficient evidence.
- **Rows:** No records will be removed during cleaning.

In [44]:
# Apply confirmed city corrections

sellers["seller_city"] = sellers["seller_city"].replace(city_mapping)

In [45]:
unicode_city_mask = (
    (sellers["seller_zip_code_prefix"] == 4557) &
    (sellers["seller_state"] == "SP") &
    (sellers["seller_city"].apply(normalize_city) == "sao paulo")
)

sellers.loc[unicode_city_mask, "seller_city"] = "sao paulo"

In [46]:
print(
    "Remaining mapped city values:",
    sellers["seller_city"].isin(city_mapping.keys()).sum()
)

print(
    "Unicode case after cleaning:",
    sellers.loc[
        sellers["seller_zip_code_prefix"] == 4557,
        "seller_city"
    ].tolist()
)

Remaining mapped city values: 0
Unicode case after cleaning: ['sao paulo']


In [47]:
print("Shape:", sellers.shape)

print("\nMissing values:")
print(sellers.isna().sum())

print("\nDuplicate rows:", sellers.duplicated().sum())

print("\nSeller IDs:")
print("Unique seller IDs:", sellers["seller_id"].nunique())
print("Duplicated seller IDs:", sellers["seller_id"].duplicated().sum())

Shape: (3095, 4)

Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Duplicate rows: 0

Seller IDs:
Unique seller IDs: 3095
Duplicated seller IDs: 0


In [48]:
city = sellers["seller_city"]

print("Leading/trailing spaces:", city.ne(city.str.strip()).sum())
print("Multiple spaces:", city.str.contains(r"\s{2,}", regex=True).sum())
print("Contains uppercase:", city.str.contains(r"[A-Z]", regex=True).sum())
print("Empty/whitespace-only:", city.str.strip().eq("").sum())

Leading/trailing spaces: 0
Multiple spaces: 0
Contains uppercase: 0
Empty/whitespace-only: 0


In [49]:
remaining_special_cities = sorted(
    city[
        city.str.contains(r"[^a-z\s]", regex=True)
    ].unique()
)

print("Unique city values with special characters:")
print(len(remaining_special_cities))

remaining_special_cities

Unique city values with special characters:
4


['andira-pr',
 "arraial d'ajuda (porto seguro)",
 "santa barbara d'oeste",
 "sao miguel d'oeste"]

In [50]:
output_path = "../CYRV_sellers_cleaned.csv"

sellers.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: ../CYRV_sellers_cleaned.csv


In [51]:
sellers_check = pd.read_csv(output_path)

print("Shape:", sellers_check.shape)

print("\nMissing values:")
print(sellers_check.isna().sum())

print("\nDuplicate rows:", sellers_check.duplicated().sum())

print("\nUnique seller IDs:", sellers_check["seller_id"].nunique())

print("\nData types:")
print(sellers_check.dtypes)

Shape: (3095, 4)

Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Duplicate rows: 0

Unique seller IDs: 3095

Data types:
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object
